In [ ]:
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 CEA and ANSES

# Preprocessing of *L. monocytogenes* and *V. parahaemolyticus* Raman spectra with Raman-DIP for their metabolic signature study 

## Library import

In [ ]:
import os, pickle
import re
import pandas as pd

## Data formating and selection

### Raw Raman spectra import from .txt to a pandas dataframe

In [ ]:
path='data/Raman_spectra_file'  # relative path of the .txt files of raw Raman spectra

In [ ]:
# Search for the folders with .txt (or .csv) files in the previously indicated "path", and append the found folders in the folder_list list with an absolute path
folder_list = []

for local_root, _, files in os.walk(path):

    for file in files:
        full_file_path = os.path.dirname(os.path.join(local_root, file))
        if any(file.endswith(ext) for ext in ['.txt', '.csv']) and full_file_path not in folder_list:
            folder_list.append(full_file_path)

In [ ]:
# get the raw .txt files as dataframes
dataframes = []
column_names = []

for path_i, temp_path in enumerate(folder_list):
    for file in os.listdir(temp_path):
        if file.endswith('.txt'):
            try:
                local_path = os.path.join(temp_path, file)
                df_temp = pd.read_csv(local_path, sep='\t', header=None)  # load the .txt file

                # Extraction of the Y column (Raman Shift) from the loaded dataframes
                series_y = df_temp.iloc[:, 1]
                series_y.name = file.split('.')[0]

                dataframes.append(series_y)
                column_names.append(series_y.name)

            except Exception as e:
                print(f"Error when loading {file} in path index {path_i}: {e}")
        else:
            print(f"File not found or not .txt format: {file}")

# Concatenation of the dataframes in a single dataframe
selected_df = pd.concat(dataframes, axis=1)

# set the X axis for all the spectra from the first one, and rename the rows as Raman Shift
selected_df.index = df_temp.iloc[:, 0]
selected_df.index.name = 'Raman Shift'

In [ ]:
# Transposition to have the indicidual spectra as rows and the Raman shifts as column.
selected_df_T = selected_df.transpose()

## Association of data and metadata

### Function with the regex expression used in the files names with the metadata

In [ ]:
def rapport_full_df(idx):
    pattern = r'^(\d{4}-\d{2}-\d{2})_([^_]+)_([^_]+)_([^_]+)_([^_]+)_([^_]+)$'
    match = re.match(pattern, idx)
    if match:
        return match.groups()
    else:
        return [None]*6

In [ ]:
names_list = ['date', 'souche', 'deuterium', 'condition_croissance', 'temps_exposition', 'num_acq']  # list of metadata elements associated with the regex

# parsing of the dataframe
parsed = selected_df_T.index.to_series().apply(rapport_full_df)
parsed_df = pd.DataFrame(parsed.tolist(), columns=names_list, index=selected_df_T.index)

df_parsed = parsed_df.join(selected_df_T)

### Selection of the preprocessed data

#### Function for the selection of the spectra according to the metadata

In [ ]:
from src.utils import split_df_by_metadata 

liste = ["souche", "condition_croissance"]
listefixe = {"deuterium": "75D2O", 
             "temps_exposition": "expocourte", 
             "souche":["16Li100A", "L6202V1", "ScottA", "EGDe", "16Li73B", "MAN195TJ01", "18-1890", "18-1732", "19-0924", "21-2854", "ATCC17802", "20-2383"],
             "condition_croissance":["temoin", "100chauffe", "0025topax", "025topactive", "05topactive"]} 
new_dict_to_preprocess = split_df_by_metadata(df = df_parsed, split_by = liste, fixed_filters= listefixe, print_summary=False)

## Preprocessing steps

### Importing preprocessing and plot functions

In [ ]:
# preprocessing functions
from src.preprocessing import als_baseline_zero_cor_df
from src.preprocessing import remove_cosmics_multispectra
from src.preprocessing import spectral_normalization_df

# plotting functions
from src.visualization import plot_spectra_by_source
from src.visualization import interactive_pca_by_source_metadata_colored
from src.visualization import plot_group_median_dict

### Preprocessing pipeline

#### Concatenation of the dataframe's dictionnary into a large dataframe

In [ ]:
frames_merged_aligned = []
for key, df in new_dict_to_preprocess.items():
    df['source'] = key
    frames_merged_aligned.append(df)


merged_aligned_concat = pd.concat(frames_merged_aligned, axis=0, ignore_index=False)
source_col_new = merged_aligned_concat['source']
df_numeric_new = merged_aligned_concat.select_dtypes(include='number')

#### Raw spectra plotting

In [ ]:
plot_spectra_by_source(df_numeric_new, source_col_new)

#### Baseline removal

In [ ]:
als_bl_removal = als_baseline_zero_cor_df(df_numeric_new, lam=1e5, p=0.01, niter=10, zero_ref="clip", return_baseline=False)

In [ ]:
interactive_pca_by_source_metadata_colored(als_bl_removal, source_col_new, meta_index=0, pc_pair=[0, 1])

#### cosmic spikes removal

In [ ]:
cosmic_filter_multi = remove_cosmics_multispectra(als_bl_removal, z_thresh=5.0)

In [ ]:
plot_spectra_by_source(cosmic_filter_multi, source_col_new)

#### Normalization

In [ ]:
normalised = spectral_normalization_df(cosmic_filter_multi, norm='l2')

In [ ]:
interactive_pca_by_source_metadata_colored(normalised, source_col_new, meta_index=0, pc_pair=[0, 1], n_components_total=10)

#### splitting of the large dataframe into sub-dataframes in a dictionnary by source and plotting

In [ ]:
after_full_pp = {}
for key in source_col_new.unique():
    after_full_pp[key] = normalised [source_col_new == key]

In [ ]:
plot_group_median_dict(after_full_pp, title="Median spectra after full preprocessing steps")   

### Saving the dictionnary of preprocessed spectra for processing

In [ ]:
# with open("preprocess_saves/2025-12-18_12strains_als-lam1e5-p0.01-niter10-zeroClip_cosmic-multispectra-zthresh5.0_norm-l2.pkl", "wb") as f:
#    pickle.dump(after_full_pp, f)